# Contexts Transformation Test

## What is tested

A `Context` describes the initial state and the relaxation of a spin system. This notebook
checks the operations on contexts (`@`, `+` and `mars.concat`) on realistic S = 1 systems. It
also checks that the transformed density matrices are physical, and that the kinetic rate
matrix agrees with the population block of the superoperator.

## List of tests

**Simple tests** (`run_tests_simple`)

1. `test_kinetics_without_zero_field_splitting`: the kinetic matrix agrees with the superoperator for spins without zero-field splitting.

**Difficult tests** (`run_tests_difficult`)

2. `test_operations_in_general_bases`: multiplication, concatenation, density transformation and kinetic consistency in bases far from the eigenbasis.
3. `test_kinetics_with_secular_approximation`: kinetic consistency when the secular approximation is enforced.
4. `test_operations_close_to_eigenbasis`: the same checks as in test 2 at a very high field, where the bases are close to the eigenbasis.
5. `test_context_summation`: the populations, density matrices, rates and superoperators of two summed contexts add up.
6. `test_explicit_density_matrix`: a context initialized with an explicit density matrix is transformed correctly.

## Checks used by the tests

- `check_multiplication`: dimensions, the Liouville-space Kronecker sum of the superoperators, unitarity of the composite basis.
- `check_summation`: dimensions and additivity of populations, density matrices, rates and superoperators.
- `check_concatenation`: dimensions and the block-diagonal structure of populations, rates and density matrices.
- `check_density_transformation`: Hermiticity, unit trace and positivity of the density matrix.
- `check_kinetic_consistency`: the rate matrix equals the population block of the superoperator.

## How to run

Run all cells. The last two cells call `run_tests_simple()` and `run_tests_difficult()`.
Each test prints one line when it passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys
import typing as tp

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
import mars.population.transform as transform
from mars import spin_model
from mars.population.contexts import Context

dtype = torch.float64

## Helper functions

### Samples and bases

In [2]:
def create_sample(electrons: list[float], g_values: tuple, zfs_values: tp.Optional[tuple]) -> spin_model.SolidSample:
    """Create a powder sample with the given g-tensor and optional zero-field splitting (D, E) in Hz."""
    g_tensor = spin_model.Interaction(g_values, dtype=dtype)
    interactions = {}
    if zfs_values is not None:
        interactions["electron_electron"] = [(0, 0, spin_model.DEInteraction(zfs_values, dtype=dtype))]
    system = spin_model.SpinSystem(electrons=electrons, g_tensors=[g_tensor], dtype=dtype, **interactions)
    return spin_model.SolidSample(base_spin_system=system, gauss=0.001, lorentz=0.001, mesh=(4, 4), dtype=dtype)


def create_samples() -> tuple[spin_model.SolidSample, spin_model.SolidSample]:
    """Create two different S = 1 samples with zero-field splitting."""
    return (
        create_sample([1.0], (2.02, 2.04, 2.06), (200e6, 50e6)),
        create_sample([1.0], (2.02, 2.14, 2.16), (200e6, 70e6)),
    )


def get_eigen_basis(sample: spin_model.SolidSample, field: float) -> torch.Tensor:
    """Return the eigenvectors of the sample Hamiltonian at the given field."""
    F, _, _, Gz = sample.get_hamiltonian_terms()
    hamiltonian = F + Gz * torch.tensor(field, dtype=dtype)
    _, vectors = torch.linalg.eigh(hamiltonian.unsqueeze(-3))
    return vectors


def direct_sum_batched(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Return the block-diagonal direct sum of two batches of square matrices."""
    n, m = a.shape[-1], b.shape[-1]
    result = torch.zeros(*a.shape[:-2], n + m, n + m, dtype=a.dtype, device=a.device)
    result[..., :n, :n] = a
    result[..., n:, n:] = b
    return result

### Contexts

In [3]:
def rates(values: list[list[float]]) -> torch.Tensor:
    """Convert a nested list of rates to a tensor."""
    return torch.tensor(values, dtype=dtype)


def create_context_pair(
    sample_1, sample_2, basis_1, basis_2, enforce_secularity, parameters_1: dict, parameters_2: dict
) -> tuple[Context, Context]:
    """Create two contexts from their sample, basis and relaxation parameters."""
    context_1 = Context(basis=basis_1, sample=sample_1, enforce_secularity=enforce_secularity, dtype=dtype, **parameters_1)
    context_2 = Context(basis=basis_2, sample=sample_2, enforce_secularity=enforce_secularity, dtype=dtype, **parameters_2)
    return context_1, context_2


def contexts_decay_only(sample_1, sample_2, basis_1, basis_2, enforce_secularity):
    """Create two contexts with pure initial states, decay and zero thermal rates."""
    return create_context_pair(
        sample_1, sample_2, basis_1, basis_2, enforce_secularity,
        {
            "init_populations": [0.0, 0.0, 1.0],
            "decay_rates": torch.tensor([123.3, 3123, 111.0], dtype=dtype),
            "thermal_rates": torch.zeros(3, 3, dtype=dtype),
        },
        {
            "init_populations": [0.0, 1.0, 0.0],
            "decay_rates": torch.tensor([434.0, 1233.0, 4343.0], dtype=dtype),
            "thermal_rates": torch.zeros(3, 3, dtype=dtype),
        },
    )


def contexts_decay_and_thermal(sample_1, sample_2, basis_1, basis_2, enforce_secularity):
    """Create two contexts with decay and symmetric thermal rates."""
    return create_context_pair(
        sample_1, sample_2, basis_1, basis_2, enforce_secularity,
        {
            "init_populations": [0.5, 0.35, 0.15],
            "decay_rates": [76.0, 200.3, 100.2],
            "thermal_rates": rates([[0.0, 100.0, 0.0], [100.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        },
        {
            "init_populations": [0.3, 0.4, 0.3],
            "decay_rates": [111.0, 1234.0, 1232.0],
            "thermal_rates": rates([[0.0, 100.0, 0.0], [100.0, 0.0, 200.0], [0.0, 200.0, 0.0]]),
        },
    )


def contexts_one_way_thermal(sample_1, sample_2, basis_1, basis_2, enforce_secularity):
    """Create two contexts without decay; the first has one one-way thermal rate, the second has none."""
    return create_context_pair(
        sample_1, sample_2, basis_1, basis_2, enforce_secularity,
        {
            "init_populations": [0.5, 0.35, 0.15],
            "decay_rates": [0.0, 0.0, 0.0],
            "thermal_rates": rates([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 100.0, 0.0]]),
        },
        {
            "init_populations": [0.3, 0.4, 0.3],
            "decay_rates": [0.0, 0.0, 0.0],
            "thermal_rates": torch.zeros(3, 3, dtype=dtype),
        },
    )


def contexts_first_thermal_only(sample_1, sample_2, basis_1, basis_2, enforce_secularity):
    """Create two contexts without decay; only the first has a symmetric thermal rate."""
    return create_context_pair(
        sample_1, sample_2, basis_1, basis_2, enforce_secularity,
        {
            "init_populations": [0.5, 0.35, 0.15],
            "decay_rates": [0.0, 0.0, 0.0],
            "thermal_rates": rates([[0.0, 0.0, 0.0], [0.0, 0.0, 100.0], [0.0, 100.0, 0.0]]),
        },
        {
            "init_populations": [0.3, 0.4, 0.3],
            "decay_rates": [0.0, 0.0, 0.0],
            "thermal_rates": torch.zeros(3, 3, dtype=dtype),
        },
    )


def contexts_general_bases(creator, sample_1, sample_2, enforce_secularity: bool = False):
    """Return context pairs in bases that are far from the eigenbasis."""
    return [
        creator(sample_1, sample_2, "xyz", "zeeman", enforce_secularity),
        creator(sample_1, sample_2, "zfs", "multiplet", enforce_secularity),
    ]


def contexts_close_to_eigenbasis(creator, sample_1, sample_2):
    """Return context pairs in bases that are close to the eigenbasis at high field."""
    return [
        creator(sample_1, sample_2, "multiplet", "multiplet", False),
        creator(sample_1, sample_2, "product", "multiplet", False),
    ]


CREATORS_GENERAL = [contexts_decay_only, contexts_decay_and_thermal, contexts_one_way_thermal]
CREATORS_CLOSE_TO_EIGENBASIS = [contexts_first_thermal_only, contexts_decay_and_thermal, contexts_decay_only]

### Full relaxation operators

In [4]:
def get_full_superoperator(context: Context, vectors: torch.Tensor):
    """Return the sum of the thermal and driven superoperators of the context."""
    thermal = context.get_transformed_thermal_superop(vectors)
    driven = context.get_transformed_driven_superop(vectors)
    result = thermal.clone() if thermal is not None else 0
    if driven is not None:
        result += driven.clone()
    return result


def get_full_kinetic_matrix(context: Context, vectors: torch.Tensor):
    """Return the rate matrix of thermal rates, driven rates and decay, with population drain on the diagonal."""
    dim = context.spin_system_dim
    mask = 1.0 - torch.eye(dim, dtype=dtype, device=context.device)
    index = torch.arange(dim, device=context.device)

    def with_drain(matrix: torch.Tensor) -> torch.Tensor:
        """Put the negative sum of the outgoing rates on the diagonal."""
        matrix = matrix.clone()
        matrix[..., index, index] = -(matrix * mask).sum(dim=-2)
        return matrix

    thermal = context.get_transformed_thermal_rates(vectors)
    result = with_drain(thermal) if thermal is not None else 0

    driven = context.get_transformed_driven_rates(vectors)
    if driven is not None:
        result += with_drain(driven)

    decay = context.get_transformed_decay_rates(vectors)
    if decay is not None:
        result -= torch.diag_embed(decay)
    return result

## Checks

In [5]:
def check_multiplication(context_1, context_2, vectors_1, vectors_2, atol: float = 1e-6, rtol: float = 1e-5) -> None:
    """Check the dimension, the Liouville-space Kronecker sum and the composite basis of context_1 @ context_2."""
    vectors = transform.batched_kron(vectors_1, vectors_2)
    product = context_1 @ context_2

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    assert product.spin_system_dim == dim_1 * dim_2, (
        f"Dimension mismatch: {dim_1} x {dim_2} = {product.spin_system_dim}"
    )

    superop_1 = context_1.get_transformed_thermal_superop(vectors_1)
    superop_2 = context_2.get_transformed_thermal_superop(vectors_2)
    superop_product = product.get_transformed_thermal_superop(vectors)

    if superop_1 is not None and superop_2 is not None:
        # Expected superoperator: R_1 x I + I x R_2
        size_1, size_2 = dim_1 * dim_1, dim_2 * dim_2
        batch_shape = torch.broadcast_shapes(superop_1.shape[:-2], superop_2.shape[:-2])
        identity_1 = torch.eye(size_1, device=superop_1.device, dtype=superop_1.dtype).expand(batch_shape + (size_1, size_1))
        identity_2 = torch.eye(size_2, device=superop_2.device, dtype=superop_2.dtype).expand(batch_shape + (size_2, size_2))
        expected = transform.batched_kron(superop_1.expand(batch_shape + (size_1, size_1)), identity_2) + (
            transform.batched_kron(identity_1, superop_2.expand(batch_shape + (size_2, size_2)))
        )
        expected = transform.reshape_superoperator_tensor_to_kronecker_basis(expected, subsystem_dims=[dim_1, dim_2])
        assert torch.allclose(superop_product, expected, atol=atol, rtol=rtol), (
            "The superoperators do not combine as a Liouville-space Kronecker sum"
        )

    if product.basis is not None:
        basis = product.basis
        overlap = basis.conj().transpose(-2, -1) @ basis
        identity = torch.eye(basis.shape[-1], device=basis.device, dtype=basis.dtype)
        assert torch.allclose(overlap, identity.expand_as(overlap), atol=1e-5), "The composite basis is not unitary"


def check_summation(context_1, context_2, vectors, atol: float = 1e-6, rtol: float = 1e-5) -> None:
    """Check that dimensions are kept and populations, densities, rates and superoperators add up."""
    summed = context_1 + context_2

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    assert dim_1 == dim_2 == summed.spin_system_dim, (
        f"Dimension mismatch: {dim_1} + {dim_2} = {summed.spin_system_dim}"
    )

    populations = [
        context.get_transformed_init_populations(vectors, normalize=False)
        for context in (context_1, context_2, summed)
    ]
    if populations[0] is not None and populations[1] is not None:
        assert torch.allclose(populations[2], populations[0] + populations[1], atol=atol, rtol=rtol), (
            "Populations do not add up"
        )

    density_1 = context_1.get_transformed_init_density(vectors)
    density_2 = context_2.get_transformed_init_density(vectors)
    density_sum = summed.get_transformed_init_density(vectors)
    if density_1 is not None and density_2 is not None:
        assert torch.allclose(density_sum, density_1 + density_2, atol=atol, rtol=rtol), (
            "Density matrices do not add up"
        )
        assert torch.allclose(density_sum, density_sum.conj().transpose(-2, -1), atol=1e-5), (
            "The summed density matrix is not Hermitian"
        )
        traces = [torch.diagonal(density, dim1=-2, dim2=-1).sum(-1) for density in (density_1, density_2, density_sum)]
        assert torch.allclose(traces[2], traces[0] + traces[1], atol=1e-5), "The trace is not additive"

    rates_1 = context_1.get_transformed_thermal_rates(vectors)
    rates_2 = context_2.get_transformed_thermal_rates(vectors)
    rates_sum = summed.get_transformed_thermal_rates(vectors)
    if rates_1 is not None and rates_2 is not None:
        assert torch.allclose(rates_sum, rates_1 + rates_2, atol=atol, rtol=rtol), "Rate matrices do not add up"

    superop_1 = context_1.get_transformed_thermal_superop(vectors)
    superop_2 = context_2.get_transformed_thermal_superop(vectors)
    superop_sum = summed.get_transformed_thermal_superop(vectors)
    if superop_1 is not None and superop_2 is not None:
        assert torch.allclose(superop_sum, superop_1 + superop_2, atol=atol, rtol=rtol), (
            "Superoperators do not add up"
        )


def check_density_transformation(context, vectors, atol: float = 1e-6) -> None:
    """Check that the transformed density matrix is Hermitian, has unit trace and is positive."""
    if context.init_density is None and context.init_populations is None:
        return
    density = context.get_transformed_init_density(vectors)
    if density is None:
        return

    assert torch.allclose(density, density.conj().transpose(-2, -1), atol=atol), "The density matrix is not Hermitian"

    trace = torch.diagonal(density, dim1=-2, dim2=-1).sum(-1)
    assert torch.allclose(trace, torch.ones_like(trace), atol=1e-5), f"The trace is {trace.mean().item():.6f}, not 1"

    if context.spin_system_dim <= 16:
        eigenvalues = torch.linalg.eigvalsh(density.real)
        assert torch.all(eigenvalues >= -1e-8), f"Negative eigenvalue: {eigenvalues.min().item():.2e}"


def check_concatenation(context_1, context_2, vectors_1, vectors_2, atol: float = 1e-6, rtol: float = 1e-5) -> None:
    """Check the dimension and the block-diagonal structure of the concatenated context."""
    concatenated = mars.concat([context_1, context_2])
    vectors = direct_sum_batched(vectors_1, vectors_2)

    dim_1, dim_2 = context_1.spin_system_dim, context_2.spin_system_dim
    assert concatenated.spin_system_dim == dim_1 + dim_2, (
        f"Dimension mismatch: {dim_1} + {dim_2} = {concatenated.spin_system_dim}"
    )

    populations_1 = context_1.get_transformed_init_populations(vectors_1, normalize=False)
    populations_2 = context_2.get_transformed_init_populations(vectors_2, normalize=False)
    populations = concatenated.get_transformed_init_populations(vectors, normalize=False)
    if populations_1 is not None and populations_2 is not None and populations is not None:
        assert torch.allclose(populations, torch.cat([populations_1, populations_2], dim=-1), atol=atol, rtol=rtol), (
            "Populations are not concatenated correctly"
        )

    rates_1 = context_1.get_transformed_thermal_rates(vectors_1)
    rates_2 = context_2.get_transformed_thermal_rates(vectors_2)
    rates_total = concatenated.get_transformed_thermal_rates(vectors)
    if rates_1 is not None and rates_2 is not None and rates_total is not None:
        assert torch.allclose(rates_total, direct_sum_batched(rates_1, rates_2), atol=atol, rtol=rtol), (
            "Rate matrices do not form a block-diagonal structure"
        )

    density_1 = context_1.get_transformed_init_density(vectors_1)
    density_2 = context_2.get_transformed_init_density(vectors_2)
    density_total = concatenated.get_transformed_init_density(vectors)
    if density_1 is not None and density_2 is not None and density_total is not None:
        assert torch.allclose(density_total, direct_sum_batched(density_1, density_2), atol=atol, rtol=rtol), (
            "Density matrices do not form a block-diagonal structure"
        )
        for number, density in enumerate((density_1, density_2), start=1):
            trace = torch.diagonal(density, dim1=-2, dim2=-1).sum(-1)
            assert torch.allclose(trace, torch.ones_like(trace), atol=1e-5, rtol=rtol), (
                f"The trace of block {number} is not 1"
            )


def check_single_kinetics(context, vectors, atol: float, rtol: float = 1e-5) -> None:
    """Check that the rate matrix of one context equals the population block of its superoperator."""
    kinetic = get_full_kinetic_matrix(context, vectors)
    extracted = transform.extract_transition_matrix_from_superoperator(get_full_superoperator(context, vectors))
    assert torch.allclose(kinetic, extracted, atol=atol, rtol=rtol), (
        f"The rate matrix differs from the superoperator by {(kinetic - extracted).abs().max():.2e}"
    )


def check_kinetic_consistency(context_1, context_2, vectors_1, vectors_2, check_product: bool = False) -> None:
    """Check rate matrices against superoperators for single, concatenated and, optionally, product contexts."""
    check_single_kinetics(context_1, vectors_1, atol=1e-5)
    check_single_kinetics(context_2, vectors_2, atol=1e-5)

    if check_product:
        # The product mixes large and small rates, so the absolute tolerance is looser here
        check_single_kinetics(context_1 @ context_2, transform.batched_kron(vectors_1, vectors_2), atol=1e-3)

    check_single_kinetics(
        mars.concat((context_1, context_2)), direct_sum_batched(vectors_1, vectors_2), atol=1e-5
    )

## Simple tests

In [6]:
def test_kinetics_without_zero_field_splitting():
    """Check that the rate matrix equals the superoperator block for S = 1 and S = 1/2 spins without ZFS."""
    sample_1 = create_sample([1.0], (2.02, 2.04, 2.06), zfs_values=None)
    sample_2 = create_sample([0.5], (2.02, 2.04, 2.06), zfs_values=None)
    vectors_1 = get_eigen_basis(sample_1, 0.1)
    vectors_2 = get_eigen_basis(sample_2, 0.1)

    context_1 = Context(
        basis="zeeman",
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.2],
        thermal_rates=torch.zeros(3, 3, dtype=dtype),
        enforce_secularity=False,
        dtype=dtype,
    )
    context_2 = Context(
        basis="zeeman",
        sample=sample_2,
        init_populations=[0.3, 0.4],
        thermal_rates=rates([[0.0, 23.0], [23.0, 0.0]]),
        enforce_secularity=False,
        dtype=dtype,
    )

    check_kinetic_consistency(context_1, context_2, vectors_1, vectors_2)
    print("Kinetics without ZFS test passed (rate matrix equals the superoperator population block)")

## Difficult tests

In [7]:
def test_operations_in_general_bases():
    """Check multiplication, concatenation, density transformation and kinetics far from the eigenbasis."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 0.01)
    vectors_2 = get_eigen_basis(sample_2, 0.01)

    for creator in CREATORS_GENERAL:
        for context_1, context_2 in contexts_general_bases(creator, sample_1, sample_2):
            check_multiplication(context_1, context_2, vectors_1, vectors_2)
            check_concatenation(context_1, context_2, vectors_1, vectors_2)
            check_density_transformation(context_1, vectors_1)
            check_kinetic_consistency(context_1, context_2, vectors_1, vectors_2, check_product=True)
    print("General bases test passed (multiplication, concatenation, density, kinetics)")


def test_kinetics_with_secular_approximation():
    """Check that rate matrices equal the superoperator blocks when the secular approximation is enforced."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 0.01)
    vectors_2 = get_eigen_basis(sample_2, 0.01)

    for creator in CREATORS_GENERAL:
        for context_1, context_2 in contexts_general_bases(creator, sample_1, sample_2, enforce_secularity=True):
            check_kinetic_consistency(context_1, context_2, vectors_1, vectors_2)
    print("Secular kinetics test passed (rate matrix equals the superoperator population block)")


def test_operations_close_to_eigenbasis():
    """Check the context operations at a very high field, where the bases are close to the eigenbasis."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 1e6)
    vectors_2 = get_eigen_basis(sample_2, 1e6)

    for creator in CREATORS_CLOSE_TO_EIGENBASIS:
        for context_1, context_2 in contexts_close_to_eigenbasis(creator, sample_1, sample_2):
            check_multiplication(context_1, context_2, vectors_1, vectors_2)
            check_concatenation(context_1, context_2, vectors_1, vectors_2)
            check_density_transformation(context_1, vectors_1)
            check_kinetic_consistency(context_1, context_2, vectors_1, vectors_2, check_product=True)
    print("Close to eigenbasis test passed (multiplication, concatenation, density, kinetics)")


def test_context_summation():
    """Check that two contexts on the same sample add up correctly."""
    sample_1, _ = create_samples()
    vectors = get_eigen_basis(sample_1, 1e6)

    for creator in CREATORS_GENERAL:
        for context_1, context_2 in contexts_general_bases(creator, sample_1, sample_1):
            check_summation(context_1, context_2, vectors)
    print("Summation test passed (populations, densities, rates and superoperators add up)")


def test_explicit_density_matrix():
    """Check the transformation of a context initialized with an explicit density matrix."""
    sample_1, _ = create_samples()
    vectors = get_eigen_basis(sample_1, 1e6)

    density = torch.tensor(
        [[0.5, 0.1 + 0.05j, 0.0], [0.1 - 0.05j, 0.3, 0.0], [0.0, 0.0, 0.2]], dtype=torch.complex128
    )
    context = Context(
        basis="zfs",
        sample=sample_1,
        init_density=density,
        thermal_rates=rates([[0.0, 1e5, 0.0], [1e5, 0.0, 1e5], [0.0, 1e5, 0.0]]),
        dtype=dtype,
    )

    check_density_transformation(context, vectors)
    print("Explicit density matrix test passed (Hermiticity, unit trace, positivity)")

## Run the tests

In [8]:
def run_tests_simple():
    """Run the simple tests of this notebook."""
    test_kinetics_without_zero_field_splitting()
    print("\nThe simple context test passed.")


def run_tests_difficult():
    """Run the difficult tests of this notebook."""
    tests = [
        test_operations_in_general_bases,
        test_kinetics_with_secular_approximation,
        test_operations_close_to_eigenbasis,
        test_context_summation,
        test_explicit_density_matrix,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} difficult context tests passed.")

In [9]:
run_tests_simple()

Kinetics without ZFS test passed (rate matrix equals the superoperator population block)

The simple context test passed.


In [10]:
run_tests_difficult()

General bases test passed (multiplication, concatenation, density, kinetics)
Secular kinetics test passed (rate matrix equals the superoperator population block)
Close to eigenbasis test passed (multiplication, concatenation, density, kinetics)
Summation test passed (populations, densities, rates and superoperators add up)
Explicit density matrix test passed (Hermiticity, unit trace, positivity)

All 5 difficult context tests passed.
